# 01 — Tokenización con spaCy
**Autor:** Giuliano Crenna, Juan Ignacio Pace (UGR)
**Fecha:** 2026-09-03
**Descripción:** Comparación de tokenización sobre el corpus procesado.
## Parámetros
- `DATA_DIR`, `SEED`, `OUT_DIR` (papermill).


In [0]:
# %% [code]
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent))
import pandas as pd
import numpy as np
DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
SEED = int(os.environ.get("SEED", 42))
np.random.seed(SEED)
df = pd.read_parquet(DATA_DIR / "processed" / "corpus_v1.parquet").head(50)
print(df[["text_clean"]].head())


In [0]:
# %% [code]
# Tokenización con spaCy. Si el modelo no está instalado, cae a .split().
try:
    import spacy
    nlp = spacy.blank("es")
    if not nlp.pipeline:
        nlp.add_pipe("sentencizer")
except Exception as exc:
    print(f"spaCy no disponible ({exc}); uso .split()")
    nlp = None
def tokenize(t):
    if nlp is None:
        return (t or "").split()
    return [tok.text for tok in nlp(t or "")]
df["tokens"] = df["text_clean"].fillna("").apply(tokenize)
df["n_tokens"] = df["tokens"].str.len()
print(df[["text_clean", "n_tokens"]].head())


## Notas
- En la etapa 4 conviene usar `es_core_news_md` para tener POS tagging
  + NER. Por ahora, en este notebook solo se valida la pipeline de
  tokenización.
- Si el modelo BETO se usa como tokenizador (WordPiece), spaCy no es
  necesario; alcanza con `BertTokenizerFast`.
